# Exportação da camada Gold para CSV

Prepara quinze tabelas candidatas, sem dados pessoais do cadastro, para uma pasta exclusiva por tentativa no Volume autorizado. O manifesto identifica a execução Gold efetiva e só existe quando todos os CSVs foram relidos e reconciliados. Cada tabela precisa corresponder ao conjunto e à quantidade explícitos de colunas do contrato candidato; a ordem do cabeçalho pode variar, campos faltantes, extras ou duplicados bloqueiam a exportação.

Contrato candidato `gold-csv-v2`: o carregador v1 não aceita este conjunto. A migração e a carga v2 são uma entrega do Backend antes do consumo no portal/IA.

A entrega completa exige também o cadastro do titular em pacote separado: dezesseis conjuntos no total. O destino restrito é verificado antes de qualquer escrita e ambos os manifestos só são publicados após a validação dos dois pacotes.


In [ ]:
def validar_estimativas_recuperacao(df):
    import json
    from math import isfinite
    nomes = ["energia_estimada_recuperavel_kwh", "intervalo_recuperacao_inferior_kwh", "intervalo_recuperacao_superior_kwh"]
    for nome in nomes:
        if dict(df.dtypes).get(nome) != "double":
            raise ValueError("Tipo de estimativa de recuperação incompatível")
    if df.filter(F.col("validacao_recuperacao_json").isNull() | F.col("estado_estimativa_recuperacao").isNull() |
        F.col("metodo_estimativa_recuperacao").isNull()).limit(1).count():
        raise ValueError("Estimativa sem identificação do método/validação")
    registros = df.select("validacao_recuperacao_json").distinct().limit(2).collect()
    if not registros:
        return
    if len(registros) != 1 or len(registros[0][0]) > 16384:
        raise ValueError("Validação de recuperação divergente ou excessiva")
    try:
        r = json.loads(registros[0][0])
    except (ValueError, TypeError):
        raise ValueError("Relatório de recuperação inválido") from None
    if not isinstance(r, dict) or r.get("metodo") != "MEDIA_POR_HISTORICO_PREVIO_V1" or r.get("alvo") != "KWH_REGISTRADO_POR_INSPECAO":
        raise ValueError("Método/alvo de recuperação incompatível")
    validado = r.get("estado") == "VALIDADO_RETROSPECTIVAMENTE"
    if not validado and r.get("estado") not in {"BASE_INSUFICIENTE", "REPROVADO_NA_VALIDACAO"}:
        raise ValueError("Estado de validação de recuperação inválido")
    preenchida = F.col(nomes[0]).isNotNull()
    invalidos = F.col("metodo_estimativa_recuperacao") != r["metodo"]
    for nome in nomes:
        invalidos = invalidos | (F.col(nome).isNotNull() != preenchida) | F.isnan(nome) | (F.abs(F.col(nome)) == float("inf")) | (F.col(nome) < 0)
    invalidos = invalidos | (F.col(nomes[1]) > F.col(nomes[0])) | (F.col(nomes[0]) > F.col(nomes[2]))
    if validado:
        metricas = [r.get(k) for k in ["mae_modelo_kwh", "mae_referencia_kwh", "cobertura_intervalo_teste", "raio_intervalo_kwh"]]
        if any(type(v) not in (int, float) or not isfinite(v) or v < 0 for v in metricas):
            raise ValueError("Métricas de validação não finitas")
        if (metricas[1] <= 0 or metricas[0] > metricas[1]*0.95 or not 0.8 <= metricas[2] <= 1 or
            any(type(r.get(k)) is not int or r[k] < 20 for k in ["qtd_treino", "qtd_calibracao", "qtd_teste"])):
            raise ValueError("Estudo não atende aos critérios de validação")
        parametros = r.get("parametros")
        if not isinstance(parametros, dict) or set(parametros) != {"0", "1"}:
            raise ValueError("Parâmetros do estudo ausentes")
        compativel = F.lit(False)
        for v in parametros.values():
            if (not isinstance(v, dict) or set(v) != {"media", "n", "inferior", "superior"} or
                type(v["n"]) is not int or v["n"] < 20 or
                any(type(v[k]) not in (int, float) or not isfinite(v[k]) or v[k] < 0 for k in ["media", "inferior", "superior"]) or
                not v["inferior"] <= v["media"] <= v["superior"]):
                raise ValueError("Parâmetros de recuperação inválidos")
            compativel = compativel | ((F.col(nomes[0]) == v["media"]) & (F.col(nomes[1]) == v["inferior"]) & (F.col(nomes[2]) == v["superior"]))
        invalidos = invalidos | (preenchida & ~compativel)
        invalidos = invalidos | (preenchida & (F.col("estado_estimativa_recuperacao") != "ESTIMATIVA_HISTORICA_POR_INSPECAO")) | (~preenchida & (F.col("estado_estimativa_recuperacao") != "FORA_POPULACAO_VALIDADA"))
    else:
        invalidos = invalidos | preenchida | (F.col("estado_estimativa_recuperacao") != r["estado"])
    if df.filter(invalidos).limit(1).count():
        raise ValueError("Estimativa de recuperação diverge do estudo validado")

import json
import os
import re
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = "workspace"
SCHEMA_GOLD = "gold"
SCHEMA_VERSION = "gold-csv-v2"
TABELAS_GOLD = [
    "diagnostico_receita", "indicadores_dec_fec", "fila_inspecao", "visao_360_uc",
    "consumo_mensal_uc", "consumo_receita_bairro", "variacao_consumo_uc",
    "faturas_uc", "medidor_uc", "atendimentos_uc", "ocorrencias_uc", "gd_uc", "tratativas_uc", "cobertura_dominios", "decomposicao_receita",
]

COLUNAS_GOLD_ESPERADAS = {
    "diagnostico_receita": [
        "competencia", "municipio", "classe_consumo", "subgrupo_tarifario", "bandeira_tarifaria", "possui_gd",
        "receita_total_reais", "receita_com_leitura_reais", "receita_sem_leitura_reais", "consumo_total_kwh",
        "receita_base_comparavel_reais", "consumo_base_comparavel_kwh", "qtd_ucs_distintas_faturadas_no_mes",
        "qtd_ucs_com_leitura", "soma_faturamentos_mensais", "qtd_leituras_observadas", "qtd_leituras_validas",
        "ticket_medio_reais_por_uc", "consumo_medio_kwh_por_uc", "tarifa_media_efetiva",
    ],
    "indicadores_dec_fec": [
        "id_alimentador", "municipio", "competencia", "total_ucs_alimentador", "ucs_interrompidas",
        "frequencia_total_interrupcoes", "duracao_total_interrupcoes_horas", "DEC", "FEC",
    ],
    "fila_inspecao": [
        "id_uc", "municipio", "bairro", "uf", "classe", "subgrupo", "alimentador", "data_ligacao",
        "status", "possui_geracao_solar", "total_inspecoes_realizadas", "total_inspecoes_irregulares",
        "media_consumo_historico", "prioridade_inspecao", "motivo_prioridade",
        "energia_estimada_recuperavel_kwh", "intervalo_recuperacao_inferior_kwh", "intervalo_recuperacao_superior_kwh",
        "estado_estimativa_recuperacao", "metodo_estimativa_recuperacao", "validacao_recuperacao_json",
    ],
    "visao_360_uc": [
        "id_uc", "municipio", "bairro", "uf", "classe", "subgrupo", "alimentador", "data_ligacao",
        "status", "possui_gd", "total_chamados_callcenter", "media_satisfacao_atendimento",
        "total_interrupcoes_sofridas", "duracao_total_interrupcoes_horas", "media_consumo_historico",
    ],
    "consumo_mensal_uc": [
        "id_uc", "competencia", "consumo_kwh", "receita_reais", "receita_com_leitura_reais",
        "receita_sem_leitura_reais", "receita_base_comparavel_reais", "consumo_base_comparavel_kwh",
        "tem_fatura", "tem_leitura", "qtd_faturas", "qtd_leituras_observadas", "qtd_leituras_validas",
    ],
    "consumo_receita_bairro": [
        "competencia", "municipio", "bairro", "classe_consumo", "receita_total_reais",
        "receita_com_leitura_reais", "receita_sem_leitura_reais", "consumo_total_kwh",
        "receita_base_comparavel_reais", "consumo_base_comparavel_kwh", "qtd_ucs_distintas_faturadas_no_mes",
        "qtd_ucs_com_leitura", "soma_faturamentos_mensais", "qtd_leituras_observadas", "qtd_leituras_validas",
    ],
    "variacao_consumo_uc": [
        "id_uc", "municipio", "bairro", "possui_gd", "consumo_medio_inicial", "meses_com_leitura_p1",
        "consumo_medio_final", "meses_com_leitura_p2", "variacao_kwh", "variacao_pct",
        "queda_consumo_superior_30_pct",
    ],
    "faturas_uc": [
        "id_fatura", "id_uc", "mes_referencia", "data_emissao", "data_vencimento", "data_pagamento",
        "valor_energia", "tributos", "valor_total", "bandeira_tarifaria", "status", "valor_fatura_declarada_em_aberto_reais", "valor_em_aberto_reais",
        "situacao_pagamento_analitica", "flag_paga", "flag_tem_pagamento", "flag_valor_inconsistente",
        "flag_data_pagamento_inconclusiva", "flag_paga_sem_data_valida", "flag_pagamento_antes_emissao",
        "flag_pagamento_sentinela", "flag_pagamento_futuro", "flag_emissao_sentinela", "flag_emissao_futura",
        "flag_vencimento_sentinela", "flag_valor_negativo", "data_processamento_gold",
    ],
    "medidor_uc": [
        "id_medidor", "id_uc", "modelo", "tipo", "data_instalacao", "data_retirada", "status",
        "flag_instalacao_sentinela", "flag_instalacao_futura", "flag_retirada_sentinela", "flag_retirada_futura",
        "ingestao_data", "situacao_na_referencia",
    ],
    "atendimentos_uc": [
        "protocolo", "id_uc", "canal", "assunto", "subassunto", "data_abertura", "data_fechamento",
        "nota_satisfacao", "qtd_tratativas", "_abertura_futura", "_fechamento_futura", "_nota_negativa",
        "_nota_invalida", "ingestao_data",
    ],
    "ocorrencias_uc": [
        "id_ocorrencia", "id_uc", "alimentador", "tipo", "causa", "data_abertura", "data_encerramento",
        "duracao_min", "flag_duracao_incompativel", "flag_abertura_futura", "flag_encerramento_futuro",
        "ingestao_data",
    ],
}
TIPOS_COMPLEMENTOS_UC = {'gd_uc': {'id_uc': 'string', 'potencia_kwp': 'double', 'data_conexao': 'date', 'modalidade': 'string', 'flag_conexao_futura': 'boolean', 'flag_conexao_sentinela': 'boolean', 'data_processamento_gold': 'timestamp'}, 'tratativas_uc': {'protocolo': 'string', 'id_uc': 'string', 'posicao_historico': 'int', 'sequencia_origem': 'int', 'data_tratativa': 'timestamp', 'acao_categoria': 'string', 'canal': 'string', 'flag_data_invalida': 'boolean', 'flag_data_futura': 'boolean', 'flag_sequencia_invalida': 'boolean', 'data_processamento_gold': 'timestamp'}, 'cadastro_titular_uc': {'id_uc': 'string', 'nome_exibicao_titular': 'string', 'situacao_identificacao': 'string', 'data_processamento_gold': 'timestamp'}}
TIPOS_COBERTURA_DOMINIOS = {'dominio': 'string', 'competencia': 'string', 'recorte': 'string', 'natureza_temporal': 'string', 'estado_cobertura': 'string', 'qtd_registros': 'bigint', 'qtd_sem_competencia': 'bigint', 'zero_registros_confirmado': 'boolean', 'data_corte_evidencia': 'timestamp', 'referencia_evidencia': 'string', 'motivo': 'string', 'origem_janela': 'string', 'natureza_certificacao': 'string', 'reconciliacao_extracao_json': 'string', 'run_id_silver': 'string', 'lote_bronze_silver': 'string', 'data_processamento_gold': 'timestamp'}
# Contrato físico da análise de duas janelas, separado dos fatos mensais.
TIPOS_DECOMPOSICAO_RECEITA = {
    "municipio": "string", "bairro": "string", "classe_consumo": "string",
    "qtd_ucs_base_comum": "bigint", "qtd_ucs_entrantes": "bigint",
    "qtd_ucs_saintes": "bigint", "qtd_ucs_observadas": "bigint",
    "faturamento_p0_reais": "double", "faturamento_p1_reais": "double",
    "efeito_volume_reais": "double", "efeito_preco_observado_reais": "double",
    "efeito_interacao_reais": "double", "variacao_entradas_reais": "double",
    "variacao_saidas_reais": "double", "variacao_sem_leitura_fora_base_reais": "double",
    "variacao_outros_fora_base_reais": "double", "estado_decomposicao": "string",
    "variacao_faturamento_total_reais": "double", "residuo_reconciliacao_reais": "double",
    "inicio_p0": "string", "fim_p0": "string", "inicio_p1": "string", "fim_p1": "string",
    "metodo": "string", "run_id_silver": "string", "lote_bronze_silver": "string",
    "data_processamento_gold": "timestamp",
}
METODO_PRECO_VOLUME = "COORTE_UC_PRECO_OBSERVADO_VOLUME_INTERACAO_V2"
LIMITACOES_MEDIDAS_GOLD = {
    "energia_faturada_kwh": "SEM_MEDIDA_COMPROVADA",
    "valor_recebido_reais": "SEM_MEDIDA_COMPROVADA",
    "energia_recuperavel_kwh": "ESTIMATIVA_HISTORICA_CONDICIONAL",
    "fechamento_mensal": "EXIGE_DECLARACAO_VINCULADA",
}


def configurar_preco_volume(janelas):
    import re
    from datetime import date
    if not isinstance(janelas, (tuple, list)) or len(janelas) != 4 or any(not isinstance(v, str) for v in janelas):
        raise ValueError("Configuração preço-volume exige quatro competências textuais")
    if not any(janelas):
        return {"estado": "NAO_CONFIGURADA", "metodo": METODO_PRECO_VOLUME, "janelas": None}
    if not all(janelas):
        raise ValueError("Preço-volume exige as quatro competências das duas janelas")
    if any(not re.fullmatch(r"[0-9]{4}-(0[1-9]|1[0-2])", v) for v in janelas):
        raise ValueError("Configuração preço-volume exige YYYY-MM")
    datas = [date.fromisoformat(v + "-01") for v in janelas]
    if datas[0] > datas[1] or datas[2] > datas[3] or datas[1] >= datas[2]:
        raise ValueError("Janelas preço-volume invertidas ou sobrepostas")
    return {"estado": "CONFIGURADA", "metodo": METODO_PRECO_VOLUME,
            "janelas": dict(zip(["inicio_p0", "fim_p0", "inicio_p1", "fim_p1"], janelas))}


def validar_configuracao_preco_volume(config):
    if not isinstance(config, dict) or set(config) != {"estado", "metodo", "janelas"}:
        raise ValueError("Contrato de configuração preço-volume inválido")
    if config["estado"] == "NAO_CONFIGURADA":
        esperado = configurar_preco_volume(("", "", "", ""))
    elif config["estado"] == "CONFIGURADA":
        janelas = config["janelas"]
        nomes = ["inicio_p0", "fim_p0", "inicio_p1", "fim_p1"]
        if not isinstance(janelas, dict) or set(janelas) != set(nomes):
            raise ValueError("Janelas preço-volume incompatíveis")
        esperado = configurar_preco_volume(tuple(janelas[k] for k in nomes))
    else:
        raise ValueError("Estado da configuração preço-volume inválido")
    if config != esperado:
        raise ValueError("Método ou configuração preço-volume incompatível")


def validar_decomposicao_receita(df, config, run_esperado, lote_esperado, instante_esperado):
    validar_configuracao_preco_volume(config)
    tipos = {c.name: c.dataType.simpleString() for c in df.schema.fields}
    if tipos != TIPOS_DECOMPOSICAO_RECEITA or len(df.columns) != len(tipos):
        raise ValueError("Schema/tipos de decomposicao_receita divergem do contrato")
    if config["estado"] == "NAO_CONFIGURADA":
        if df.limit(1).count():
            raise ValueError("Análise não configurada não pode reutilizar resultado anterior")
        return
    chaves = ["municipio", "bairro", "classe_consumo", "inicio_p0", "fim_p0", "inicio_p1", "fim_p1"]
    obrigatorios = chaves + ["estado_decomposicao", "metodo", "run_id_silver", "lote_bronze_silver",
                            "data_processamento_gold"] + [k for k in tipos if k.startswith("qtd_")]
    invalidos = F.lit(False)
    for nome in obrigatorios:
        invalidos = invalidos | F.col(nome).isNull()
    invalidos = invalidos | (F.col("run_id_silver") != F.lit(run_esperado)) | (F.col("lote_bronze_silver") != F.lit(lote_esperado))
    invalidos = invalidos | (F.col("data_processamento_gold") != instante_esperado) | (F.col("metodo") != METODO_PRECO_VOLUME)
    for nome, valor in config["janelas"].items():
        invalidos = invalidos | (F.col(nome) != valor)
    estados = ["CALCULADA_BASE_COMUM", "SEM_BASE_COMUM", "COBERTURA_NAO_CERTIFICADA"]
    invalidos = invalidos | ~F.col("estado_decomposicao").isin(estados)
    comum = F.col("estado_decomposicao") == "CALCULADA_BASE_COMUM"
    sem = F.col("estado_decomposicao") == "SEM_BASE_COMUM"
    nao_certificada = F.col("estado_decomposicao") == "COBERTURA_NAO_CERTIFICADA"
    invalidos = invalidos | (comum & (F.col("qtd_ucs_base_comum") <= 0)) | (sem & (F.col("qtd_ucs_base_comum") != 0))
    invalidos = invalidos | (F.col("qtd_ucs_observadas") <= 0) | (
        F.col("qtd_ucs_base_comum") + F.col("qtd_ucs_entrantes") + F.col("qtd_ucs_saintes") > F.col("qtd_ucs_observadas"))
    for nome, tipo in tipos.items():
        if tipo == "bigint":
            invalidos = invalidos | (F.col(nome) < 0)
        if tipo == "double":
            invalidos = invalidos | F.isnan(nome) | (F.abs(F.col(nome)) == float("inf"))
            invalidos = invalidos | (nao_certificada & F.col(nome).isNotNull())
            if nome.startswith("efeito_"):
                invalidos = invalidos | (comum & F.col(nome).isNull()) | (sem & F.col(nome).isNotNull())
            else:
                invalidos = invalidos | (~nao_certificada & F.col(nome).isNull())
    termos = ["efeito_volume_reais", "efeito_preco_observado_reais", "efeito_interacao_reais",
              "variacao_entradas_reais", "variacao_saidas_reais", "variacao_sem_leitura_fora_base_reais",
              "variacao_outros_fora_base_reais"]
    soma = F.lit(0.)
    for nome in termos:
        soma = soma + F.coalesce(F.col(nome), F.lit(0.))
    tolerancia = F.greatest(F.lit(0.000001), F.abs("variacao_faturamento_total_reais") * F.lit(1e-9))
    residuo = F.col("variacao_faturamento_total_reais") - soma
    invalidos = invalidos | (~nao_certificada & ((F.abs(residuo) > tolerancia) |
        (F.abs(residuo-F.col("residuo_reconciliacao_reais")) > tolerancia) |
        (F.abs(F.col("faturamento_p1_reais")-F.col("faturamento_p0_reais")-F.col("variacao_faturamento_total_reais")) > tolerancia)))
    if df.filter(invalidos).limit(1).count() or df.groupBy(*chaves).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("Análise preço-volume com identidade, valores ou reconciliação inválidos")


TIPOS_GOLD_COMPLEMENTARES = {**TIPOS_COMPLEMENTOS_UC, "cobertura_dominios": TIPOS_COBERTURA_DOMINIOS, "decomposicao_receita": TIPOS_DECOMPOSICAO_RECEITA}
for nome in ["gd_uc", "tratativas_uc", "cobertura_dominios", "decomposicao_receita"]:
    COLUNAS_GOLD_ESPERADAS[nome] = list(TIPOS_GOLD_COMPLEMENTARES[nome])
SCHEMA_CADASTRO = "gold_cadastro"

if set(COLUNAS_GOLD_ESPERADAS) != set(TABELAS_GOLD):
    raise RuntimeError("Inventário do exportador diverge do contrato candidato")

def validar_colunas_gold_exportadas(nome_tabela, df):
    esperadas = COLUNAS_GOLD_ESPERADAS.get(nome_tabela)
    recebidas = list(df.columns)
    faltantes = sorted(set(esperadas or []) - set(recebidas))
    extras = sorted(set(recebidas) - set(esperadas or []))
    duplicadas = sorted({coluna for coluna in recebidas if recebidas.count(coluna) > 1})
    if esperadas is None or len(recebidas) != len(esperadas) or faltantes or extras or duplicadas:
        raise RuntimeError(
            f"Schema de {nome_tabela} diverge do contrato candidato: "
            f"faltantes={faltantes}; extras={extras}; colunas_duplicadas={duplicadas}"
        )

    tipos_esperados = TIPOS_GOLD_COMPLEMENTARES.get(nome_tabela)
    if tipos_esperados and {campo.name: campo.dataType.simpleString() for campo in df.schema.fields} != tipos_esperados:
        raise RuntimeError(f"Tipos de {nome_tabela} divergem do contrato")

    if nome_tabela == "fila_inspecao":
        validar_estimativas_recuperacao(df)
    if nome_tabela == "decomposicao_receita":
        validar_decomposicao_receita(df, controle_exportacao["preco_volume_configuracao"],
            controle_exportacao["run_id_silver"], controle_exportacao["lote_bronze_silver"],
            F.lit(controle_exportacao["preco_volume_referencia"]).cast("timestamp"))

def validar_cadastro_restrito(df):
    tipos = {campo.name: campo.dataType.simpleString() for campo in df.schema.fields}
    if len(df.columns) != len(tipos) or tipos != TIPOS_COMPLEMENTOS_UC["cadastro_titular_uc"]:
        raise RuntimeError("Schema/tipos do cadastro restrito divergem do contrato")

dbutils.widgets.text("ambiente", "", "Ambiente (opcional, somente prod)")
dbutils.widgets.text("run_id", "", "Run Gold esperado (opcional)")
dbutils.widgets.dropdown("exportar_cadastro_restrito", "sim", ["sim"], "Cadastro restrito obrigatório na entrega completa")
if dbutils.widgets.get("exportar_cadastro_restrito").strip().lower() != "sim":
    raise RuntimeError("A entrega exige o cadastro restrito; atualizar configuração antiga para sim")
ambiente_param = dbutils.widgets.get("ambiente").strip().lower()
run_id_param = dbutils.widgets.get("run_id").strip()
if ambiente_param and ambiente_param != "prod":
    raise RuntimeError("Exportação oficial exige ambiente prod.")

def ler_controle():
    df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.controle_gold")
    rows = (df.filter(F.col("status_gold") == "CONCLUIDO")
            .orderBy(F.col("processado_em").desc()).limit(2)
            .withColumn("_instante", F.date_format("processado_em", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))
            .collect())
    if not rows or (len(rows) == 2 and rows[0]["processado_em"] == rows[1]["processado_em"]):
        raise RuntimeError("Controle Gold ausente ou ambíguo")
    row = rows[0].asDict()
    required = ["run_id_gold", "run_id_silver", "lote_bronze_silver", "_instante"]
    if any(not isinstance(row.get(k), str) or not row[k].strip() for k in required):
        raise RuntimeError("Controle Gold sem identidade completa")
    if (str(row.get("ambiente", "")).strip().lower() != "prod"
            or row.get("silver_lineage_status") != "CONTROLE_FISICO"
            or row.get("status_silver") != "CONCLUIDO"):
        raise RuntimeError("Controle Gold não comprova publicação produtiva validada")
    generated = [t.strip() for t in str(row.get("tabelas_geradas", "")).split(",")]
    if len(generated) != len(TABELAS_GOLD) or set(generated) != set(TABELAS_GOLD):
        raise RuntimeError("Controle Gold não contempla as quinze tabelas analíticas contratadas")
    if row.get("tabelas_restritas_geradas") != "cadastro_titular_uc":
        raise RuntimeError("Controle Gold sem inventário cadastral restrito compatível")
    # Mesmos formatos de origem do contrato histórico; não aceitar IDs manuais/de reserva.
    for campo, prefixo in (("run_id_gold", "GOLD_"), ("run_id_silver", ""), ("lote_bronze_silver", "")):
        match = re.fullmatch(prefixo + r"([0-9]{8}T[0-9]{6}Z)_[0-9a-f]{8}", row[campo])
        if match is None:
            raise RuntimeError("Controle Gold contém identificador de origem inválido")
        try:
            datetime.strptime(match.group(1), "%Y%m%dT%H%M%SZ")
        except ValueError:
            raise RuntimeError("Controle Gold contém instante de origem inválido") from None
    def objeto_sem_repeticao(pares):
        objeto = {}
        for chave, valor in pares:
            if chave in objeto:
                raise ValueError("Campo repetido no controle da análise")
            objeto[chave] = valor
        return objeto
    try:
        config_texto = row.get("preco_volume_configuracao_json")
        limites_texto = row.get("limitacoes_medidas_json")
        if not isinstance(config_texto, str) or not isinstance(limites_texto, str) or len(config_texto) > 2048 or len(limites_texto) > 2048:
            raise ValueError("Metadados de análise ausentes ou excessivos")
        config = json.loads(config_texto, object_pairs_hook=objeto_sem_repeticao)
        limites = json.loads(limites_texto, object_pairs_hook=objeto_sem_repeticao)
        validar_configuracao_preco_volume(config)
        if limites != LIMITACOES_MEDIDAS_GOLD:
            raise ValueError("Limitações das medidas incompatíveis")
        referencia = row.get("preco_volume_referencia")
        if not isinstance(referencia, str):
            raise ValueError("Referência da análise ausente")
        instante = datetime.fromisoformat(referencia.replace("Z", "+00:00"))
        if instante.tzinfo is None or instante.utcoffset() != timezone.utc.utcoffset(instante):
            raise ValueError("Referência da análise exige UTC explícito")
        if instante > datetime.fromisoformat(row["_instante"].replace("Z", "+00:00")):
            raise ValueError("Referência da análise posterior à conclusão Gold")
    except (ValueError, TypeError):
        raise RuntimeError("Controle Gold sem configuração/referência/limitações compatíveis") from None
    row["preco_volume_configuracao"] = config
    row["limitacoes_medidas"] = limites
    row.pop("processado_em")
    return row

controle_exportacao = ler_controle()
run_id_real = controle_exportacao["run_id_gold"]
if not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", run_id_real):
    raise RuntimeError("Identidade Gold inválida para o destino")
if run_id_param and run_id_param != run_id_real:
    raise RuntimeError("Run solicitado diverge da publicação Gold validada")

def versao_delta(nome):
    schema = SCHEMA_CADASTRO if nome == "cadastro_titular_uc" else SCHEMA_GOLD
    return int(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{schema}.{nome} LIMIT 1").first()["version"])

# Job Gold e exportação devem executar em sequência, sem execuções concorrentes.
# A versão de cada tabela é fixada antes de qualquer escrita de CSV.
versoes_delta = {nome: versao_delta(nome) for nome in TABELAS_GOLD}
versoes_delta["cadastro_titular_uc"] = versao_delta("cadastro_titular_uc")
if ler_controle() != controle_exportacao:
    raise RuntimeError("Publicação Gold mudou antes da exportação")


In [ ]:
# O destino cadastral deve estar disponível antes de iniciar qualquer exportação.
base_restrita = f"/Volumes/{CATALOG}/{SCHEMA_CADASTRO}/cadastro_export_volume/export_csv"
try:
    dbutils.fs.ls(base_restrita.rsplit("/", 1)[0])
except Exception as erro:
    raise RuntimeError("Destino cadastral restrito indisponível; preparar Volume e permissões antes da entrega") from erro

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}.gold_export_volume")
PATH_EXPORT_BASE = f"/Volumes/{CATALOG}/{SCHEMA_GOLD}/gold_export_volume/export_csv"
export_id = uuid4().hex
PATH_EXPORT = f"{PATH_EXPORT_BASE}/{run_id_real}/{export_id}"
os.makedirs(PATH_EXPORT, exist_ok=False)
cadastro_export_id = uuid4().hex
path_cadastro = f"{base_restrita}/{run_id_real}/{cadastro_export_id}"
os.makedirs(path_cadastro, exist_ok=False)


In [ ]:
# Nulo tem representação distinta de string vazia; o marcador literal é reservado.
CSV_OPTIONS = {
    "header": "true", "lineSep": "\n", "sep": ";", "encoding": "UTF-8", "quoteAll": "true", 'multiLine': 'true', 'escape': '"', 'quote': '"',
    "nullValue": "\\N", "emptyValue": "", "dateFormat": "yyyy-MM-dd",
    "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX", "timeZone": "UTC",
}
def exportar_tabela_csv(nome, schema, versao, path_export, validar):
    df = (spark.read.option("versionAsOf", versao)
          .table(f"{CATALOG}.{schema}.{nome}"))
    validar(df)
    for campo in df.schema.fields:
        if campo.dataType.simpleString() == "string" and df.filter(F.col(campo.name) == CSV_OPTIONS["nullValue"]).limit(1).count():
            raise ValueError("Fonte contém marcador reservado de nulo CSV")
    destino = f"{path_export}/{nome}"
    df.write.format("csv").mode("errorifexists").options(**CSV_OPTIONS).save(destino)
    relido = (spark.read.schema(df.schema).options(**CSV_OPTIONS)
              .option("mode", "FAILFAST").option("enforceSchema", "false").csv(destino))
    if df.exceptAll(relido).limit(1).count() or relido.exceptAll(df).limit(1).count():
        raise RuntimeError(f"CSV divergente da Gold: {nome}")
    return {"registros": df.count(), "versao_delta": versao, "schema": df.schema.jsonValue()}

controles_tabelas = {}
for nome in TABELAS_GOLD:
    controles_tabelas[nome] = exportar_tabela_csv(
        nome, SCHEMA_GOLD, versoes_delta[nome], PATH_EXPORT,
        lambda df, nome=nome: validar_colunas_gold_exportadas(nome, df))

if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
    raise RuntimeError("Gold mudou durante a exportação; tentativa sem manifesto")


## Manifesto da exportação oficial


### Validação e publicação do manifesto

Exige controle produtivo com linhagem física, quinze tabelas e run esperado compatível. Fixam-se versões Delta e conferem-se controle/versões ao terminar. Cada tentativa usa diretórios exclusivos nos dois Volumes; falha capturada remove os manifestos da tentativa atual e preserva exportações anteriores. Não consumir pastas sem manifesto.

O Job deve serializar Gold → exportação e impedir execuções concorrentes: a promoção Gold de quinze tabelas não é uma transação única. Os CSVs são relidos com o schema original e comparados como multiconjuntos antes da liberação. Datas usam ISO; timestamps UTC com microssegundos e offset; decimais preservam escala; `\N` representa nulo e string vazia permanece distinta. O contrato v2 requer adaptação explícita do carregador, sem alterar o snapshot v1.


In [ ]:
# Revalidar imediatamente antes de publicar o manifesto.
if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
    raise RuntimeError("Publicação Gold mudou; manifesto não será liberado")
manifesto = {
    "schema_version": SCHEMA_VERSION, "ambiente": "prod",
    "lote": run_id_real, "run_id": run_id_real,
    "run_id_silver": controle_exportacao["run_id_silver"],
    "lote_bronze_silver": controle_exportacao["lote_bronze_silver"],
    "data_processamento_gold": controle_exportacao["_instante"],
    "status_gold": controle_exportacao["status_gold"],
    "data_exportacao_csv": datetime.now(timezone.utc).isoformat(),
    "export_id": export_id, "tabelas_exportadas": list(TABELAS_GOLD),
    "cadastro_export_id": cadastro_export_id, "cadastro_schema_version": "gold-cadastro-v1",
    "controles_tabelas": controles_tabelas, "formato_csv": CSV_OPTIONS,
    "preco_volume_configuracao": controle_exportacao["preco_volume_configuracao"],
    "preco_volume_referencia": controle_exportacao["preco_volume_referencia"],
    "limitacoes_medidas": controle_exportacao["limitacoes_medidas"],
    "responsavel": "Equipe de Engenharia de Dados (Gold)",
}
caminho_manifesto = os.path.join(PATH_EXPORT, "manifest.json")
def publicar_manifesto(caminho, dados):
    conteudo_manifesto = json.dumps(dados, indent=2, ensure_ascii=False)
    manifesto_temporario = caminho + ".tmp"
    try:
        with open(manifesto_temporario, "x", encoding="utf-8") as arquivo:
            arquivo.write(conteudo_manifesto)
        os.replace(manifesto_temporario, caminho)
    except Exception:
        if os.path.exists(manifesto_temporario):
            os.remove(manifesto_temporario)
        raise

# Publicação adiada até verificar também o CSV cadastral na etapa seguinte.
# Nenhum manifesto desta tentativa é liberado nesta célula.


## Operação e recuperação

- Executar depois do sucesso do Job Gold, com exclusão de execuções concorrentes. Usar o caminho da tentativa concluída, informado em `PATH_EXPORT`; não misturar arquivos de diretórios diferentes.
- Uma falha antes do manifesto exige uma nova tentativa. Não promover manualmente arquivos parciais nem sobrescrever o manifesto da exportação anterior.
- O run vem exclusivamente de `controle_gold`; o parâmetro opcional somente confirma essa identidade. Não existe fallback manual de lote.
- O manifesto carrega a execução Gold, a origem Silver/Bronze e versões físicas. Os controles e dados reais permanecem no Volume autorizado.
- `gold-csv-v2` é candidato até validar o Job real e integrar migração/carga v2. O Backend v1 deve rejeitá-lo explicitamente; esta entrega não ativa consumidores nem conclui WEB-03.

## Cadastro do titular — etapa obrigatória da entrega

O pacote analítico tem quinze tabelas e não inclui nomes. A entrega exige também `cadastro_titular_uc` de `gold_cadastro`, em Volume separado previamente provisionado. `exportar_cadastro_restrito=sim` é padrão e único valor aceito; configuração antiga com `nao` interrompe antes da escrita. Ambos os pacotes revalidam a mesma execução/versões. Seus manifestos só são liberados após os dezesseis conjuntos serem reconciliados, com referências cruzadas; falha interrompe a entrega e recupera manifestos da tentativa atual. Não misturar pacotes nem usar nome sintético como fallback.


In [ ]:
# Cadastro obrigatório, em Volume separado e sem nomes nos logs.
# O Volume e as permissões cadastrais são provisionados pelo operador autorizado.
if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
    raise RuntimeError("Gold mudou antes da exportação restrita")
controle_cadastro = exportar_tabela_csv("cadastro_titular_uc", SCHEMA_CADASTRO,
    versoes_delta["cadastro_titular_uc"], path_cadastro, validar_cadastro_restrito)
if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
    raise RuntimeError("Gold mudou durante a exportação restrita; manifesto não será liberado")
manifesto_cadastro = {
    "schema_version": "gold-cadastro-v1", "ambiente": "prod", "restrito": True,
    "finalidade": "identificacao_titular_atendimento", "run_id": run_id_real,
    "run_id_silver": controle_exportacao["run_id_silver"],
    "lote_bronze_silver": controle_exportacao["lote_bronze_silver"],
    "data_processamento_gold": controle_exportacao["_instante"],
    "data_exportacao_csv": datetime.now(timezone.utc).isoformat(),
    "status_gold": controle_exportacao["status_gold"], "export_id": cadastro_export_id,
    "analytic_export_id": export_id, "analytic_schema_version": SCHEMA_VERSION,
    "tabelas_exportadas": ["cadastro_titular_uc"],
    "controles_tabelas": {"cadastro_titular_uc": controle_cadastro}, "formato_csv": CSV_OPTIONS,
}
# Não há transação entre Volumes. Publicar cadastro primeiro e analítico por último;
# o consumidor exige sucesso do Job e referências cruzadas entre ambos os manifestos.
caminho_manifesto_cadastro = os.path.join(path_cadastro, "manifest.json")
try:
    publicar_manifesto(caminho_manifesto_cadastro, manifesto_cadastro)
    if ler_controle() != controle_exportacao or any(versao_delta(t) != v for t,v in versoes_delta.items()):
        raise RuntimeError("Gold mudou durante a publicação dos pacotes")
    publicar_manifesto(caminho_manifesto, manifesto)
except Exception as erro:
    falha_recuperacao = False
    # Somente manifestos desta tentativa exclusiva; exportações anteriores são preservadas.
    for caminho in [caminho_manifesto, caminho_manifesto_cadastro]:
        try:
            if os.path.exists(caminho):
                os.remove(caminho)
        except Exception:
            falha_recuperacao = True
    if falha_recuperacao:
        raise RuntimeError("Falha ao recuperar manifestos parciais; bloquear consumo desta tentativa") from erro
    raise
print("[PASS] Entrega completa validada: 15 projeções analíticas e 1 cadastral em pacotes separados.")
